### 567. Permutation in String
Given two strings `s1` and `s2`, return `true` if `s2` contains a **permutation** of `s1`, or `false` otherwise.

In other words, return `true` if one of `s1`'s permutations is the substring of `s2`.

**Example 1:**
```shell
Input: s1 = "ab", s2 = "eidbaooo"
Output: true
```
**Explanation:** `s2` contains one permutation of `s1` (`"ba"`).

**Example 2:**
```shell
Input: s1 = "ab", s2 = "eidboaoo"
Output: false
```

**Constraints:**

- `1 <= s1.length, s2.length <= 104`
- `s1` and `s2` consist of lowercase English letters.

---

### Reasoning
**Step 1**: restate the problem in your own words and identify what you're looking for. What does a valid window in `s2` look like, and what's its size?
- I should find out whether `s2` contains a substring of `s1` which could be a permutation, but still a substring. I think if `s2` contains `s1` as a substring it's also True, the point is to keep `s1` as a substring of `s2` either a permutation version or not. I think also if `s2` is exactly `s1` or a permutation of `s1` it's also valid. (Or we only deal with permutations of `s1` in `s2`, and that is only valid case?)
A valid window in `s2` is any sequence of chars that is a permutation of `s1`, with the same length with `s1`.

You got it at the end:
> a valid window is **any substring of `s2` with the exact same length as `s1` whose character frequencies match `s1`**. A non-permuted exact match counts too (it's technically a permutation).

The crucial observation you just made:
> **the window size is fixed at `len(s1)`.** That's what makes this different from the last two problems — the window doesn't grow and shrink based on a condition. It slides at a **fixed size**.

That's your biggest hint for the approach.

**Step 2 — trace one example by hand.** Work through `s1 = "ab"`, `s2 = "eidbaooo"` manually. List every window of size 2 in `s2` and check each one. What are you comparing, and what's the first one that matches?
- All windows of size 2 in `s2`: `ei`, `id`, `db`, `ba`, `ao`, `oo`, `oo`. We actually scan every `s2` window and check membership for each character in the given `s1`. This way, we will find that a window `'ba'` is a permutation of `s1` within `s2`.

Correct windows, correct match. But "check membership" isn't precise enough — membership just checks if a character _exists_. What if `s1 = "aab"` and a window is `"abb"`? Both contain `a` and `b`, but they're not permutations.

What do you actually need to compare between `s1` and the window?
- Looks like I need not only checking membership in `s1`, but the **exact occurance of each character** with `.count()` method. This will limit to the exact number of char's occurances in a substing not relying on their order at all what is needed for permutation check.

Exactly.
> Frequency match = permutation. Order doesn't matter, counts do.

**Step 3 — brute force.** You just described it: slide every window of size `len(s1)` across `s2`, and for each window compare character counts against `s1`'s counts.

What's the time complexity of that brute force? Count the operations:
- how many windows are there, and
- what does each comparison cost?

  - sliding across each `s2` windows will take $O(n)$ since it is `len(s2) - 1 = 7` windows. Windows are overlapping. Then, for each window we use `.count()` which probably takes also $O(n)$ (but it might be optimized by Python to $O(1)$, since it's a build-in one, I'm not sure yet). If not optimized it would be total cost of $O(n^2)$, the comparison itself is $O(1)$, so per each window there is $O(n)$ for counting chars.

`.count()` is $O(n)$ regardless of being built-in — "built-in" just means implemented in C, not that the algorithm changes. It still scans every character. So brute force is **$O(n × k)$** where `n` = `len(s2)` and `k` = `len(s1)` — effectively $O(n²)$ when they're similar lengths.

**Step 4 — the optimization.** You already know the key insight from the previous two problems: **don't recompute from scratch on each step**. When the window slides one position right, what's the minimal update needed to the frequency counts — and why does that make each step $O(1)$?
- We might use a dict data structure to store the current window's chars mapping to their counts. Looks like when the current window slides to the right the next char is what must be checked and counted. So this next char is added mapping to it's count whick is still $O(n)$ with `.count()`, and the old left most is deleted from the dict which is $O(1)$.

Close — but you're still thinking about `.count()` for the new character, which scans the whole string again. You don't need to count anything from scratch.

Think about what's already in the dict when the window slides one step. You have the counts for the current window. The new window differs by exactly two characters: one added on the right, one removed on the left.

So instead of counting, what two $O(1)$ dict operations get you the updated frequency map?
- In dict, lookup, insert and delete are $O(1)$ operations. I need to add the next element to the window, and delete the old left most. Have the counts for the current window, with the left most it's clear, deletion takes $O(1)$. When I add the next new item I just count it as 1 in the current window which is also $O(1)$. This way I have updated current window with counts to compare againsts `s1`, and it will cost $O(1)$ for the step.

Exactly right.
> Two $O(1)$ dict operations per slide.

Now the comparison: you have two dicts — `s1_counts` and `window_counts`. What does that comparison cost, and is it truly $O(1)$?
- When comparing two dicts, if they have the same number of the same keys mapping to the same values, the dicts are equal. No matter in wich order they are. I'm not sure this comparison costs $O(1)$, since we need to scan each key under the hood to make sure they are the same with the same values. If it is the case, the comparison will be $O(n)$, and we fall back to $O(n^2)$ in total.

Good reasoning.
> dict comparison IS $O(k)$ where `k` is the **number of keys**.

But what is `k` bounded by in this specific problem?

> Both strings contain only lowercase English letters. That's at **most 26 possible keys** in either dict — always, regardless of how long `s1` or `s2` are.

> So the comparison is $O(26)$ = **$O(1)$** — bounded by the **alphabet size**, not the input size. A constant upper bound stays constant **no matter how large `n` grows**.

**This distinction matters in interviews**:

```shell
O(n) — cost grows with input size         → not constant
O(26) — cost bounded by fixed alphabet    → treated as O(1)
```

**Full complexity**:

- **Time**:  $O(n)$  — n slides, each $O(1)$ update + $O(26)$ comparison
- **Space**: $O(1)$  — two dicts, each at most 26 keys regardless of input size


Now write the code. You have everything:

1. Build `s1_counts` once
2. Build initial `window_counts` for the first window
3. Slide: add right char, remove left char, compare

Go ahead — write the function signature and the skeleton first before filling in the logic.

[Visual](../images/567permutation_in_str.png)

In [ ]:
def permutation_in_str(s1: str, s2: str) -> bool:
    # sliding window - fixed margins
    # initial setup:
    # set s1_counts dict
    # set window_counts dict of len(s1)
    # set l = 0
    # slide window across s2 -> [len(s1), len(s2)]

    # compare dicts - initial & current window check
    # window slide:
    # build window dict - add/increment r bound; evict/decrease l bound
    # return if dicts equal - last window check

    # k - number of keys in dict
    if len(s1) > len(s2):
        return False

    s1_counts = {c: s1.count(c) for c in s1}                         # O(k^2)
    k = len(s1)
    window_init = s2[:k]                                             # init window  
    window_counts = {c: window_init.count(c) for c in window_init}   # O(k^2)
    l = 0  # used to move window left boundary

    for r in range(k, len(s2)):
        # initial & current window check
        if s1_counts == window_counts:
            return True
        # 1. add/increment window r bound
        window_counts[s2[r]] = window_counts.get(s2[r], 0) + 1
        # 2. evict/decrease window l bound
        if window_counts[s2[l]] <= 1:
            window_counts.pop(s2[l])
        else:
            window_counts[s2[l]] -= 1
        l += 1
    # last window check
    return s1_counts == window_counts

In [15]:
permutation_in_str("ab", "eidbaooo")  # True
# permutation_in_str("ab", "eidboaoo")  # False
# permutation_in_str("aa", "baa")  # True
# permutation_in_str("ab", "aab")  # True

True

The solution is correct.

One small note: building `s1_counts` and `window_counts` using `.count()` inside a dict comprehension is technically $O(k²)$ for the initial setup — it calls `.count()` once per unique character. Using `Counter` is cleaner and $O(k)$:

```python
from collections import Counter
s1_counts = Counter(s1)
window_counts = Counter(s2[:len(s1)])
```

Functionally identical, but `Counter` is the idiomatic Python tool for this exact job. Worth knowing for interviews.

In [ ]:
from collections import Counter

def permutation_in_str2(s1: str, s2: str) -> bool:
    if len(s1) > len(s2):
        return False

    s1_counts = Counter(s1)                         # O(k)
    k = len(s1)
    window_init = s2[:k]
    window_counts = Counter(window_init)            # O(k)
    l = 0

    for r in range(len(s1), len(s2)):      # move window right boundary
        # initial & current window check
        if s1_counts == window_counts:
            return True
        # 1. add/increment window r bound
        window_counts[s2[r]] = window_counts.get(s2[r], 0) + 1
        # 2. evict/decrease window l bound
        if window_counts[s2[l]] <= 1:
            window_counts.pop(s2[l]) 
        else:
            window_counts[s2[l]] -= 1
        # 3. move window left boundary
        l += 1
    # last window check
    return s1_counts == window_counts

In [20]:
permutation_in_str2("ab", "eidbaooo")  # True
# permutation_in_str2("ab", "eidboaoo")  # False
# permutation_in_str2("aa", "baa")  # True
# permutation_in_str2("ab", "aab")  # True

True

#### Sliding Window — pattern locked

File this:

> **FIXED window**: size determined by `s1`, slides by 1 each step → build initial window, then add right / remove left
> **VARIABLE window**: size determined by a **condition** (sum, uniqueness) → expand always, shrink while condition violated

**Both**:
- $O(n)$ time — each element added **once**, removed **once**
- Each step is $O(1)$ — **no recomputation** from scratch
- Dict comparison bounded by alphabet size → $O(1)$

**When to recognize it**:
- "substring/subarray satisfying a condition" or,
- "does X contain Y as a permutation/anagram"
